# Field Proposal Review Risk Model PoC

기존 합성 상담 시나리오에서 로컬로 생성한 150행 CSV를 검증하고 Dummy, Logistic Regression, Random Forest를 학습·평가합니다. 원천 JSONL과 실제 개인정보는 Colab에 업로드하지 않습니다. `needs_review`와 `confidence`는 제출용 PoC를 위해 파생·모사한 값이며 실서비스 성능을 의미하지 않습니다.

In [ ]:
# @title 1. 실행 경로 및 라이브러리 확인
from pathlib import Path
import os
import importlib.util
import platform
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import joblib

local_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
candidates = [Path(os.environ.get('F2_POC_ROOT', '')), Path('/content/f2_poc'), local_root, local_root / 'ml/field_proposal_reliability']
POC_ROOT = next((path.resolve() for path in candidates if path and (path / 'data/synthetic_field_proposals.csv').is_file() and (path / 'scripts/run_experiment.py').is_file()), None)
if POC_ROOT is None:
    raise FileNotFoundError('PoC root를 찾지 못했습니다. /content/f2_poc에 CSV와 run_experiment.py를 업로드하거나 F2_POC_ROOT를 설정하세요.')
print('PoC root:', POC_ROOT)
try:
    ON_GOOGLE_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    ON_GOOGLE_COLAB = False
RUNTIME_LABEL = 'google_colab_cpu' if ON_GOOGLE_COLAB else 'local_notebook_verification'
print('Runtime label:', RUNTIME_LABEL)
print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('matplotlib:', matplotlib.__version__)
print('joblib:', joblib.__version__)

In [ ]:
# @title 2. 데이터 기본 통계 확인
dataset_path = POC_ROOT / 'data/synthetic_field_proposals.csv'
df = pd.read_csv(dataset_path)
print('전체 행 수:', len(df))
print('컬럼 목록:', df.columns.tolist())
print('Target 분포:')
print(df['needs_review'].value_counts().sort_index())
print('field_type 분포:')
print(df['field_type'].value_counts().sort_index())
print('결측치 개수:')
print(df.isna().sum())
print('첫 10행:')
display(df.head(10))

In [ ]:
# @title 3. 모델 학습, 평가, 저장
import subprocess
import sys
command = [sys.executable, str(POC_ROOT / 'scripts/run_experiment.py'), '--poc-root', str(POC_ROOT), '--dataset', str(dataset_path), '--execution-label', RUNTIME_LABEL]
completed = subprocess.run(command, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
completed.check_returncode()

In [ ]:
# @title 4. 실제 결과 확인 및 다운로드 묶음 생성
import json
import shutil
metrics = json.loads((POC_ROOT / 'reports/metrics.json').read_text(encoding='utf-8'))
comparison = pd.read_csv(POC_ROOT / 'reports/model_comparison.csv')
display(comparison)
print('Final Model:', metrics['final_model']['name'])
print('선정 이유:', metrics['final_model']['selection_reason'])
archive_base = Path('/content/f2_poc_outputs') if ON_GOOGLE_COLAB else Path('/tmp/f2_poc_outputs')
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=POC_ROOT)
print('다운로드 파일:', archive)